In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key='put your key here')
project = rf.workspace("ml-ssuvd").project("3d-printing-defect-detection-evopu")
version = project.version(2)
dataset = version.download("yolo26")


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26l.pt") 

results = model.train(
    data=f"/kaggle/working/3D-Printing-Defect-Detection-2", 
    epochs=30,
    imgsz=640,             
    batch=32,              
    device="0,1",
    patience=5,          
    cache=False,                       
    save_period=10,
    project="/kaggle/working/training", 
    name="model_V1" 
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from ultralytics import YOLO

# --- 1. Evaluate the Final Best Model ---
final_model_path = "/kaggle/working/training/model_V1-3/weights/best.pt"

# UPDATE THIS LINE: Point directly to the .yaml file
data_path = "/kaggle/working/3D-Printing-Defect-Detection-2/data.yaml" 

print("Evaluating final model...")
model = YOLO(final_model_path)
metrics = model.val(data=data_path)

print("\n--- Final Model Metrics ---")
print(f"Overall mAP50-95: {metrics.box.map:.4f}")
print(f"Overall mAP50:    {metrics.box.map50:.4f}")
print(f"Precision:        {metrics.box.mp:.4f}")
print(f"Recall:           {metrics.box.mr:.4f}")

# --- 2. Combine results.csv from all 3 runs ---
run_dirs = [
    "/kaggle/working/training/model_V1",
    "/kaggle/working/training/model_V1-2",
    "/kaggle/working/training/model_V1-3"
]

combined_df = pd.DataFrame()

for run_dir in run_dirs:
    csv_path = os.path.join(run_dir, "results.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip() 
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    else:
        print(f"Warning: Could not find {csv_path}")

total_epochs = len(combined_df)
if total_epochs > 0:
    combined_df['continuous_epoch'] = range(1, total_epochs + 1)
    print(f"\nTotal epochs trained across all runs: {total_epochs}")

    combined_csv_path = "/kaggle/working/training/combined_results.csv"
    combined_df.to_csv(combined_csv_path, index=False)
    print(f"Saved combined CSV to: {combined_csv_path}")

    # --- 3. Plot Combined Metrics and Losses ---
    columns_to_plot = [
        'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
        'val/box_loss', 'val/cls_loss', 'val/dfl_loss',
        'metrics/precision(B)', 'metrics/recall(B)',
        'metrics/mAP50(B)', 'metrics/mAP50-95(B)'
    ]

    actual_cols = [col for col in columns_to_plot if col in combined_df.columns]

    plt.figure(figsize=(18, 12))
    plt.suptitle(f"Continuous Training Progress ({total_epochs} Total Epochs)", fontsize=18, fontweight='bold')

    for i, col in enumerate(actual_cols, 1):
        plt.subplot(3, 4, i)
        color = '#1f77b4' if 'train' in col else '#ff7f0e' if 'loss' in col else '#2ca02c'
        plt.plot(combined_df['continuous_epoch'], combined_df[col], label=col, color=color, linewidth=2)
        plt.title(col.split('/')[-1].replace('(B)', ''), fontsize=12)
        plt.xlabel('Total Epochs')
        plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.subplots_adjust(top=0.92) 

    plot_path = "/kaggle/working/training/combined_training_graphs.png"
    plt.savefig(plot_path, dpi=300)
    print(f"Saved combined training graph to: {plot_path}")
    plt.show()
else:
    print("No CSV data found to plot. Check your run directories.")
    

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil
from ultralytics import YOLO

# --- 1. Evaluate the Final Best Model ---
final_model_path = "/kaggle/working/training/model_V1-3/weights/best.pt"
data_path = "/kaggle/working/3D-Printing-Defect-Detection-2/data.yaml" 

print("Evaluating final model...")
model = YOLO(final_model_path)
metrics = model.val(data=data_path)

print("\n--- Final Model Metrics ---")
print(f"Overall mAP50-95: {metrics.box.map:.4f}")
print(f"Overall mAP50:    {metrics.box.map50:.4f}")
print(f"Precision:        {metrics.box.mp:.4f}")
print(f"Recall:           {metrics.box.mr:.4f}")

# --- Create an export directory for clean zipping ---
export_dir = "/kaggle/working/training_export"
os.makedirs(export_dir, exist_ok=True)

# --- 2. Combine results.csv from all 3 runs ---
run_dirs = [
    "/kaggle/working/training/model_V1",
    "/kaggle/working/training/model_V1-2",
    "/kaggle/working/training/model_V1-3"
]

combined_df = pd.DataFrame()

for run_dir in run_dirs:
    csv_path = os.path.join(run_dir, "results.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        # Strip leading spaces from Ultralytics CSV headers
        df.columns = df.columns.str.strip() 
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    else:
        print(f"Warning: Could not find {csv_path}")

total_epochs = len(combined_df)
if total_epochs > 0:
    # Add continuous epoch column
    combined_df['continuous_epoch'] = range(1, total_epochs + 1)
    print(f"\nTotal epochs trained across all runs: {total_epochs}")

    # Save CSV to export folder
    combined_csv_path = os.path.join(export_dir, "combined_results.csv")
    combined_df.to_csv(combined_csv_path, index=False)
    print(f"Saved combined CSV to: {combined_csv_path}")

    # --- 3. Plot Combined Metrics and Losses ---
    columns_to_plot = [
        'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
        'val/box_loss', 'val/cls_loss', 'val/dfl_loss',
        'metrics/precision(B)', 'metrics/recall(B)',
        'metrics/mAP50(B)', 'metrics/mAP50-95(B)'
    ]

    actual_cols = [col for col in columns_to_plot if col in combined_df.columns]

    # 3A. Save Combined High-Quality Grid Plot
    print("\nGenerating high-quality combined grid plot...")
    plt.figure(figsize=(20, 14))
    plt.suptitle(f"Continuous Training Progress ({total_epochs} Total Epochs)", fontsize=24, fontweight='bold')

    for i, col in enumerate(actual_cols, 1):
        plt.subplot(3, 4, i)
        color = '#1f77b4' if 'train' in col else '#ff7f0e' if 'loss' in col else '#2ca02c'
        plt.plot(combined_df['continuous_epoch'], combined_df[col], label=col, color=color, linewidth=2.5)
        plt.title(col.split('/')[-1].replace('(B)', ''), fontsize=16, fontweight='bold')
        plt.xlabel('Total Epochs', fontsize=12)
        plt.ylabel('Value', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.subplots_adjust(top=0.92) 

    # Save with 600 DPI for high resolution
    combined_plot_path = os.path.join(export_dir, "combined_training_graphs_grid.png")
    plt.savefig(combined_plot_path, dpi=600, bbox_inches='tight')
    plt.close() # Close to free memory
    
    # 3B. Save Individual High-Quality Plots for each metric
    print("Generating high-quality individual metric plots...")
    for col in actual_cols:
        plt.figure(figsize=(10, 6))
        color = '#1f77b4' if 'train' in col else '#ff7f0e' if 'loss' in col else '#2ca02c'
        
        plt.plot(combined_df['continuous_epoch'], combined_df[col], label=col, color=color, linewidth=3)
        
        clean_name = col.split('/')[-1].replace('(B)', '')
        plt.title(f"{clean_name} over {total_epochs} Epochs", fontsize=18, fontweight='bold')
        plt.xlabel('Continuous Epoch', fontsize=14)
        plt.ylabel(clean_name, fontsize=14)
        plt.grid(True, linestyle='--', alpha=0.7)
        
        # Create safe filename without slashes
        safe_filename = col.replace('/', '_').replace('(', '').replace(')', '') + "_high_res.png"
        indiv_plot_path = os.path.join(export_dir, safe_filename)
        
        plt.savefig(indiv_plot_path, dpi=600, bbox_inches='tight')
        plt.close()
    
    # --- 4. Create ZIP archive for easy download ---
    zip_base_path = "/kaggle/working/training_results_export"
    shutil.make_archive(zip_base_path, 'zip', export_dir)
    print(f"\n✅ All done! Successfully zipped your high-res graphs and CSV into: {zip_base_path}.zip")
    print("👉 Look in the right-hand panel under '/kaggle/working/'. You can download 'training_results_export.zip' directly to your computer.")

else:
    print("No CSV data found to plot. Check your run directories.")
    

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Evaluating final model...
Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26l summary (fused): 190 layers, 24,749,595 parameters, 0 gradients, 86.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1386.9±427.8 MB/s, size: 50.0 KB)
val: Scanning /kaggle/working/3D-Printing-Defect-Detection-2/valid/labels.cache... 1527 images, 393 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1527/1527 376.7Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 209, len(boxes) = 6304. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a 

In [ ]:
import kagglehub

handle = 'soumyacode8/3d-printing-defect-detection-yolo26l-training'
local_dataset_dir = '/kaggle/working/training'

# Create a new dataset
# kagglehub.dataset_upload(handle, local_dataset_dir)

# You can then create a new version of this dataset and include version notes.
kagglehub.dataset_upload(handle, local_dataset_dir, version_notes='improved data')


In [ ]:
import os
os.remove("/kaggle/working/training/dataset-metadata.json")


In [ ]:
import shutil
shutil.rmtree("/kaggle/working/training/dataset-metadata.json")


In [ ]:
# import shutil

# def zip_folder(folder_path, output_path):
#     shutil.make_archive(output_path, 'zip', folder_path)
#     print(f"Successfully zipped '{folder_path}' into '{output_path}.zip'")

# folder_to_zip = "/kaggle/working/training" 
# output_zip_name = "/kaggle/working/run" 
# zip_folder(folder_to_zip, output_zip_name)


In [ ]:
import os
import random
import cv2
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ultralytics import YOLO

# --- Paths ---
images_dir = "/kaggle/working/3D-Printing-Defect-Detection-2/valid/images"
labels_dir = "/kaggle/working/3D-Printing-Defect-Detection-2/valid/labels"
model_path = "/kaggle/working/training/model_V3/weights/best.pt"

# Load the trained model
model = YOLO(model_path)
class_names = model.names 

# --- Helper function to draw Ground Truth boxes ---
def draw_ground_truth(image_path, label_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) < 5: continue
                
                cls_id = int(parts[0])
                x_center, y_center, box_w, box_h = map(float, parts[1:5])
                
                x1 = int((x_center - box_w / 2) * w)
                y1 = int((y_center - box_h / 2) * h)
                x2 = int((x_center + box_w / 2) * w)
                y2 = int((y_center + box_h / 2) * h)
                
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                label = class_names.get(cls_id, str(cls_id))
                cv2.putText(img, label, (x1, max(15, y1 - 5)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return img

# --- Select 10 Random Images ---
all_images = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
selected_images = random.sample(all_images, min(10, len(all_images)))

# --- Plotting with Plotly for Interactive Zoom ---
for i, img_name in enumerate(selected_images):
    img_path = os.path.join(images_dir, img_name)
    
    label_name = os.path.splitext(img_name)[0] + ".txt"
    label_path = os.path.join(labels_dir, label_name)
    
    # 1. Get Ground Truth Image
    gt_img = draw_ground_truth(img_path, label_path)
    
    # 2. Get Predicted Image
    results = model(img_path, verbose=False)
    pred_img = results[0].plot() 
    pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB) 
    
    # 3. Create Interactive Subplots
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Real / Ground Truth", "Model Prediction"))
    
    fig.add_trace(go.Image(z=gt_img), row=1, col=1)
    fig.add_trace(go.Image(z=pred_img), row=1, col=2)
    
    # Hide axis ticks for a cleaner view
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    
    # Update layout to be a good viewing size
    fig.update_layout(
        height=450, 
        width=950, 
        margin=dict(l=10, r=10, t=30, b=10),
        hovermode=False # Disables the pixel-color tooltip so it doesn't block the view
    )
    
    fig.show()
    